# CNN-EXP-038 — LOW_TP score-matched residual test

## Главный вопрос

> Можно ли поднять LOW_TP относительно миллионов отрицательных с похожим
> исходным score, **не разрушая уже правильное ранжирование EXP036**?

EXP037 заменял presence-head целиком и разрушил global ranking.
Здесь старая WIDTH64 presence-head остаётся якорем.

```text
EXP036 frozen backbone
        ↓
64-D nucleotide features ──→ linear residual correction c(x)

EXP036 old presence logit s₀
        ↓
s_final = s₀ + α · c(x)
```

### Что обучаем

Только **линейную residual correction**:

```text
64 → 1
```

Положительные:
- LOW_TP train: нижний квартиль positives по старому EXP036 presence score.

Отрицательные:
- score-matched natural TN из тех же диапазонов старого score;
- большой representative random-TN background.

### Что НЕ делаем

- не меняем WIDTH64 backbone;
- не меняем RF=1029;
- не переобучаем старую presence-head;
- не трогаем intensity-head;
- не делаем LOW_TP oversampling внутри CNN;
- не добавляем MLP;
- не делаем WIDTH128.

### Cross-contig CV

```text
8 train-contigs → обучаем residual correction
1 train-contig  → полный natural-background AP
```

Подбирается только:

```text
α = 0, 0.1, 0.25, 0.5, 1.0
```

`α=0` означает исходный EXP036 и обязан участвовать в выборе.

Primary selection metric:
- mean natural-background AP по held-out train-contigs.

Диагностика:
- recovered LOW_TP;
- new FP / recovered LOW_TP;
- сохранение HIGH_TP;
- Spearman.


In [1]:
from pathlib import Path
import gc
import json
import math
import sys
import time

import numpy as np
import pandas as pd
from IPython.display import display

import torch
from torch import nn
import torch.nn.functional as F
import torch_directml

from scipy.stats import pearsonr, rankdata

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = next(
    candidate
    for candidate in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (candidate / "README.md").is_file()
    and (candidate / "src/ml_project").is_dir()
)

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project.epic_baseline import load_baseline_sequences
from ml_project.epic_cnn import (
    ProfileWindowConfig,
    ProfileWindowGenerator,
    create_cnn_presence_intensity,
)
from ml_project.epic_data import load_split

device = torch_directml.device()

print("Project:", PROJECT_ROOT)
print("DirectML:", device)
print("GPU:", torch_directml.device_name(0))


Project: D:\Projects\EPIC\EPIC
DirectML: privateuseone:0
GPU: AMD Radeon RX 6750 GRE 12GB 


## 1. Загружаем frozen EXP036 и feature bank EXP037

EXP037 уже один раз извлёк:

- все train TP features;
- representative random TN features;
- baseline WIDTH64 scores.

Мы используем этот bank повторно, чтобы EXP038 не делал лишний дорогой
feature-extraction pass для обучения residual head.

**HARD_FP из EXP037 здесь не используются.**


In [2]:
split = load_split(PROJECT_ROOT)
sequences, fasta_alphabet = load_baseline_sequences(split)

profile_config = ProfileWindowConfig(
    target_length=1024,
    context_flank=130,
    positive_branch_fraction=0.5,
)

EXP036_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP036_WIDTH64_RF1029"
    / "run_001"
)

EXP037_RUN = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / "CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER"
    / "run_001"
)

EXP036_CHECKPOINT = EXP036_RUN / "width64_final.pt"
TRAIN_FEATURE_BANK = EXP037_RUN / "train_feature_bank_width64.npz"

assert EXP036_CHECKPOINT.is_file(), EXP036_CHECKPOINT
assert TRAIN_FEATURE_BANK.is_file(), (
    f"Не найден EXP037 feature bank: {TRAIN_FEATURE_BANK}\n"
    "EXP038 специально переиспользует его."
)

checkpoint = torch.load(
    EXP036_CHECKPOINT,
    map_location="cpu",
    weights_only=False,
)

frozen_model = create_cnn_presence_intensity()
frozen_model.load_state_dict(checkpoint["model_state_dict"])

assert frozen_model.backbone.stem.out_channels == 64
assert frozen_model.profile_head.in_channels == 64
assert frozen_model.profile_head.out_channels == 2

rf = 1
for block in frozen_model.backbone.blocks:
    for conv in (block.conv1, block.conv2):
        rf += (int(conv.kernel_size[0]) - 1) * int(conv.dilation[0])

assert rf == 1029

for parameter in frozen_model.parameters():
    parameter.requires_grad_(False)

frozen_model = frozen_model.to(device)
frozen_model.eval()

print("Checkpoint:", EXP036_CHECKPOINT)
print("Feature bank:", TRAIN_FEATURE_BANK)
print("WIDTH:", frozen_model.backbone.stem.out_channels)
print("RF:", rf)
print("Frozen:", all(not p.requires_grad for p in frozen_model.parameters()))


Checkpoint: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP036_WIDTH64_RF1029\run_001\width64_final.pt
Feature bank: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP037_FROZEN_WIDTH64_HARDFP_RERANKER\run_001\train_feature_bank_width64.npz
WIDTH: 64
RF: 1029
Frozen: True


In [3]:
train_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="train",
    config=profile_config,
    seed=42,
)

validation_generator = ProfileWindowGenerator(
    prepared_split=split,
    sequences=sequences,
    part="validation",
    config=profile_config,
    seed=42,
)

train_tiles = train_generator.tile_index.copy().reset_index(drop=True)
validation_tiles = validation_generator.tile_index.copy().reset_index(drop=True)

TRAIN_CONTIGS = sorted(train_tiles["contig"].astype(str).unique())
VALIDATION_CONTIGS = sorted(validation_tiles["contig"].astype(str).unique())

print("Train contigs:", len(TRAIN_CONTIGS), TRAIN_CONTIGS)
print("Validation contigs:", len(VALIDATION_CONTIGS), VALIDATION_CONTIGS)
print("Train tiles:", len(train_tiles))
print("Validation tiles:", len(validation_tiles))


Train contigs: 9 ['NC_064035.1', 'NC_064036.1', 'NC_064038.1', 'NC_064039.1', 'NC_064040.1', 'NC_064043.1', 'NC_064045.1', 'NC_064047.1', 'NC_064048.1']
Validation contigs: 3 ['NC_064034.1', 'NC_064041.1', 'NC_064042.1']
Train tiles: 130525
Validation tiles: 47743


In [4]:
bank = np.load(TRAIN_FEATURE_BANK, allow_pickle=False)

pos_X = bank["pos_X"].astype(np.float32, copy=False)
pos_old_score = bank["pos_old_score"].astype(np.float32, copy=False)
pos_count = bank["pos_count"].astype(np.float32, copy=False)
pos_contig = bank["pos_contig"]

random_X = bank["random_X"].astype(np.float32, copy=False)
random_old_score = bank["random_old_score"].astype(np.float32, copy=False)
random_contig = bank["random_contig"]

assert pos_X.shape[1] == 64
assert random_X.shape[1] == 64

print("All TP:", pos_X.shape)
print("Representative random TN:", random_X.shape)

display(
    pd.DataFrame({
        "contig": TRAIN_CONTIGS,
        "TP": [int(np.sum(pos_contig == c)) for c in TRAIN_CONTIGS],
        "random_TN": [int(np.sum(random_contig == c)) for c in TRAIN_CONTIGS],
    })
)


All TP: (170443, 64)
Representative random TN: (1044002, 64)


,contig,TP,random_TN
0,NC_064035.1,30987,151318
1,NC_064036.1,26323,142686
2,NC_064038.1,14793,117192
3,NC_064039.1,21975,123984
4,NC_064040.1,19067,118478
5,NC_064043.1,19558,109086
6,NC_064045.1,15184,95862
7,NC_064047.1,11668,101970
8,NC_064048.1,10888,83426


## 2. Residual training helpers

LOW_TP определяется **внутри каждой CV fold только по 8 training contigs**:

```text
LOW_TP = bottom 25% old EXP036 score среди train positives
```

Score-matched TN строятся из representative natural TN pool:

1. LOW_TP score делим на quantile bins.
2. В каждом bin берём TN из того же диапазона baseline score.
3. Число TN в bin пропорционально числу LOW_TP в этом bin.

Отдельно добавляется большой random-TN background.

Так correction вынуждена искать сигнал, который существует **сверх старого score**.


In [5]:
EXPERIMENT_ID = "CNN-EXP-038"
EXPERIMENT_SLUG = "CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL"

RUN_DIR = (
    PROJECT_ROOT
    / "artifacts"
    / "experiments"
    / EXPERIMENT_SLUG
    / "run_001"
)
RUN_DIR.mkdir(parents=True, exist_ok=True)

LOW_TP_QUANTILE = 0.25

MATCH_BINS = 12
MATCHED_TN_PER_LOWTP = 2.0
RANDOM_TN_PER_LOWTP = 4.0

HEAD_EPOCHS = 50
HEAD_BATCH_SIZE = 8192
HEAD_LR = 3e-3
HEAD_WEIGHT_DECAY = 1e-4
HEAD_SEED = 20260924

ALPHAS = (0.0, 0.1, 0.25, 0.5, 1.0)

SUBMISSION_LEVELS = 100_001

print("Artifacts:", RUN_DIR)
print("LOW_TP quantile:", LOW_TP_QUANTILE)
print("ALPHAS:", ALPHAS)


Artifacts: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001
LOW_TP quantile: 0.25
ALPHAS: (0.0, 0.1, 0.25, 0.5, 1.0)


In [6]:
def sigmoid_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    out = np.empty_like(x, dtype=np.float64)

    nonnegative = x >= 0
    out[nonnegative] = 1.0 / (1.0 + np.exp(-x[nonnegative]))

    exp_x = np.exp(x[~nonnegative])
    out[~nonnegative] = exp_x / (1.0 + exp_x)

    return out.astype(np.float32)


def safe_logit(probability, eps=1e-6):
    p = np.clip(
        np.asarray(probability, dtype=np.float64),
        eps,
        1.0 - eps,
    )
    return np.log(p / (1.0 - p)).astype(np.float32)


def fit_scaler(X):
    mean = X.mean(axis=0, dtype=np.float64).astype(np.float32)
    std = X.std(axis=0, dtype=np.float64).astype(np.float32)
    std = np.maximum(std, 1e-6)
    return mean, std


def build_score_matched_negatives(
    low_scores,
    negative_scores,
    *,
    multiplier,
    n_bins,
    rng,
):
    low_scores = np.asarray(low_scores, dtype=np.float64)
    negative_scores = np.asarray(negative_scores, dtype=np.float64)

    # Quantile bins follow LOW_TP score distribution.
    quantiles = np.linspace(0.0, 1.0, n_bins + 1)
    edges = np.quantile(low_scores, quantiles)
    edges = np.unique(edges)

    # Degenerate fallback.
    if len(edges) < 3:
        lo = float(np.min(low_scores))
        hi = float(np.max(low_scores))
        if hi <= lo:
            hi = lo + 1e-6
        edges = np.linspace(lo, hi, 3)

    selected = []

    for bin_i in range(len(edges) - 1):
        left = edges[bin_i]
        right = edges[bin_i + 1]

        if bin_i == len(edges) - 2:
            low_mask = (low_scores >= left) & (low_scores <= right)
            neg_mask = (negative_scores >= left) & (negative_scores <= right)
        else:
            low_mask = (low_scores >= left) & (low_scores < right)
            neg_mask = (negative_scores >= left) & (negative_scores < right)

        n_low = int(low_mask.sum())
        if n_low == 0:
            continue

        candidates = np.flatnonzero(neg_mask)

        # If a narrow bin has no negatives, widen locally around the bin.
        if len(candidates) == 0:
            center = 0.5 * (left + right)
            distance = np.abs(negative_scores - center)
            fallback_n = min(
                max(int(round(multiplier * n_low)), 32),
                len(negative_scores),
            )
            candidates = np.argpartition(
                distance,
                fallback_n - 1,
            )[:fallback_n]

        target_n = max(
            1,
            int(round(multiplier * n_low)),
        )

        chosen = rng.choice(
            candidates,
            size=target_n,
            replace=target_n > len(candidates),
        )
        selected.append(chosen)

    if not selected:
        raise RuntimeError("Не удалось собрать score-matched TN.")

    return np.concatenate(selected)


def train_residual_correction(
    *,
    train_pos_X,
    train_pos_score,
    train_random_X,
    train_random_score,
    seed,
):
    rng = np.random.default_rng(seed)

    low_cutoff = float(
        np.quantile(
            train_pos_score,
            LOW_TP_QUANTILE,
        )
    )

    low_mask = train_pos_score <= low_cutoff

    low_X = train_pos_X[low_mask]
    low_score = train_pos_score[low_mask]

    matched_idx = build_score_matched_negatives(
        low_score,
        train_random_score,
        multiplier=MATCHED_TN_PER_LOWTP,
        n_bins=MATCH_BINS,
        rng=rng,
    )

    matched_X = train_random_X[matched_idx]

    n_random_background = max(
        1,
        int(round(RANDOM_TN_PER_LOWTP * len(low_X))),
    )

    random_idx = rng.choice(
        len(train_random_X),
        size=n_random_background,
        replace=n_random_background > len(train_random_X),
    )

    background_X = train_random_X[random_idx]

    X_negative = np.concatenate(
        (matched_X, background_X),
        axis=0,
    )

    # Standardization fitted only on the fold's training cohort.
    scaler_source = np.concatenate(
        (low_X, X_negative),
        axis=0,
    )
    mean, std = fit_scaler(scaler_source)
    del scaler_source

    X_positive_std = ((low_X - mean) / std).astype(np.float32, copy=False)
    X_negative_std = ((X_negative - mean) / std).astype(np.float32, copy=False)

    X = np.concatenate(
        (X_positive_std, X_negative_std),
        axis=0,
    )

    y = np.concatenate(
        (
            np.ones(len(X_positive_std), dtype=np.float32),
            np.zeros(len(X_negative_std), dtype=np.float32),
        )
    )

    correction = nn.Linear(64, 1, bias=True)

    # Residual starts as zero correction.
    with torch.no_grad():
        correction.weight.zero_()
        correction.bias.zero_()

    optimizer = torch.optim.AdamW(
        correction.parameters(),
        lr=HEAD_LR,
        weight_decay=HEAD_WEIGHT_DECAY,
    )

    pos_weight = torch.tensor(
        [len(X_negative_std) / max(len(X_positive_std), 1)],
        dtype=torch.float32,
    )

    criterion = nn.BCEWithLogitsLoss(
        pos_weight=pos_weight
    )

    for epoch in range(HEAD_EPOCHS):
        order = rng.permutation(len(X))

        for start in range(0, len(X), HEAD_BATCH_SIZE):
            idx = order[start:start + HEAD_BATCH_SIZE]

            xb = torch.from_numpy(X[idx])
            yb = torch.from_numpy(y[idx])[:, None]

            optimizer.zero_grad(set_to_none=True)
            logit = correction(xb)
            loss = criterion(logit, yb)
            loss.backward()
            optimizer.step()

    w_std = (
        correction.weight[0]
        .detach().cpu().numpy()
        .astype(np.float32, copy=True)
    )
    b_std = float(correction.bias[0].detach().cpu().item())

    # Fold feature scaling into a raw-feature linear correction.
    w_raw = (w_std / std).astype(np.float32)
    b_raw = float(
        b_std - np.dot(w_raw, mean)
    )

    # Normalize correction scale using representative train random TN.
    # This makes alpha comparable across CV folds.
    background_correction = (
        train_random_X @ w_raw + b_raw
    ).astype(np.float64)

    corr_mean = float(background_correction.mean())
    corr_std = float(background_correction.std())

    if corr_std < 1e-6:
        corr_std = 1.0

    w_raw = (w_raw / corr_std).astype(np.float32)
    b_raw = float((b_raw - corr_mean) / corr_std)

    return {
        "w_raw": w_raw,
        "b_raw": b_raw,
        "low_cutoff": low_cutoff,
        "n_low_tp": int(len(low_X)),
        "n_matched_tn": int(len(matched_X)),
        "n_random_tn": int(len(background_X)),
        "correction_background_mean_before_norm": corr_mean,
        "correction_background_std_before_norm": corr_std,
    }


def correction_raw(features, head):
    return (
        features @ head["w_raw"]
        + head["b_raw"]
    ).astype(np.float32)


## 3. Metric helpers

AP считаем по 5-decimal submission bins с tie grouping.

LOW_TP/HIGH_TP diagnostics определяются **baseline EXP036 score**,
чтобы reranker не мог сам переопределить удобные группы.


In [7]:
def probability_bin_5dp(probabilities):
    return np.rint(
        np.clip(probabilities, 0.0, 1.0) * 100_000
    ).astype(np.int32)


def update_histogram(hist, indices):
    values, counts = np.unique(indices, return_counts=True)
    hist[values] += counts.astype(np.int64)


def ap_from_histograms(positive_hist, total_hist):
    positives = int(positive_hist.sum())
    if positives <= 0:
        return float("nan")

    positive_desc = positive_hist[::-1]
    total_desc = total_hist[::-1]

    tp = np.cumsum(positive_desc, dtype=np.int64)
    predicted_positive = np.cumsum(total_desc, dtype=np.int64)

    precision = tp / np.maximum(predicted_positive, 1)

    return float(
        np.sum(
            (positive_desc / positives) * precision
        )
    )


def dense_rank_pearson(prediction, target):
    prediction = np.asarray(prediction)
    target = np.asarray(target)

    if len(prediction) < 2:
        return float("nan")

    return float(
        pearsonr(
            rankdata(prediction, method="dense"),
            rankdata(target, method="dense"),
        )[0]
    )


def extract_backbone_and_old_outputs(model, batch):
    x_gpu = batch.x_both_strands.to(device)

    feature_gpu = model.backbone(x_gpu)
    output_gpu = model.profile_head(feature_gpu)

    c = profile_config.context_flank
    L = profile_config.target_length

    feature = (
        feature_gpu[:, :, c:c + L]
        .permute(0, 2, 1)
        .detach().cpu().numpy()
        .astype(np.float32, copy=True)
    )

    output = (
        output_gpu[:, :, c:c + L]
        .detach().cpu().numpy()
        .astype(np.float32, copy=True)
    )

    old_logit = output[:, 0, :]
    intensity = output[:, 1, :]

    del x_gpu, feature_gpu, output_gpu

    return feature, old_logit, intensity


def make_eval_state():
    return {
        alpha: {
            "total_hist": np.zeros(SUBMISSION_LEVELS, dtype=np.int64),
            "positive_hist": np.zeros(SUBMISSION_LEVELS, dtype=np.int64),
            "positive_score": [],
            "positive_count": [],
        }
        for alpha in ALPHAS
    }


## 4. Cross-contig CV

Для каждой fold обучаем **одну correction head** на 8 contigs.

Затем на полном natural-background held-out contig одновременно оцениваем все α:

```text
final_logit = old_logit + α * correction
```

`α=0` — EXP036 baseline.


In [8]:
cv_heads = {}

for fold_i, heldout_contig in enumerate(TRAIN_CONTIGS, 1):
    print()
    print("=" * 80)
    print(
        f"TRAIN RESIDUAL FOLD {fold_i}/{len(TRAIN_CONTIGS)} "
        f"heldout={heldout_contig}"
    )
    print("=" * 80)

    pos_train_mask = pos_contig != heldout_contig
    random_train_mask = random_contig != heldout_contig

    head = train_residual_correction(
        train_pos_X=pos_X[pos_train_mask],
        train_pos_score=pos_old_score[pos_train_mask],
        train_random_X=random_X[random_train_mask],
        train_random_score=random_old_score[random_train_mask],
        seed=HEAD_SEED + fold_i,
    )

    cv_heads[heldout_contig] = head

    print(
        {
            "LOW_TP": head["n_low_tp"],
            "matched_TN": head["n_matched_tn"],
            "random_TN": head["n_random_tn"],
            "LOW_TP_cutoff": head["low_cutoff"],
        }
    )



TRAIN RESIDUAL FOLD 1/9 heldout=NC_064035.1
{'LOW_TP': 34864, 'matched_TN': 69728, 'random_TN': 139456, 'LOW_TP_cutoff': 0.602599024772644}

TRAIN RESIDUAL FOLD 2/9 heldout=NC_064036.1
{'LOW_TP': 36030, 'matched_TN': 72060, 'random_TN': 144120, 'LOW_TP_cutoff': 0.6006374955177307}

TRAIN RESIDUAL FOLD 3/9 heldout=NC_064038.1
{'LOW_TP': 38913, 'matched_TN': 77826, 'random_TN': 155652, 'LOW_TP_cutoff': 0.602820873260498}

TRAIN RESIDUAL FOLD 4/9 heldout=NC_064039.1
{'LOW_TP': 37117, 'matched_TN': 74234, 'random_TN': 148468, 'LOW_TP_cutoff': 0.6023373007774353}

TRAIN RESIDUAL FOLD 5/9 heldout=NC_064040.1
{'LOW_TP': 37844, 'matched_TN': 75688, 'random_TN': 151376, 'LOW_TP_cutoff': 0.6022781133651733}

TRAIN RESIDUAL FOLD 6/9 heldout=NC_064043.1
{'LOW_TP': 37722, 'matched_TN': 75444, 'random_TN': 150888, 'LOW_TP_cutoff': 0.6025576591491699}

TRAIN RESIDUAL FOLD 7/9 heldout=NC_064045.1
{'LOW_TP': 38815, 'matched_TN': 77630, 'random_TN': 155260, 'LOW_TP_cutoff': 0.6045618057250977}

TRAIN R

In [9]:
CV_RESULTS_PATH = RUN_DIR / "cross_contig_cv.csv"

cv_rows = []

frozen_model.eval()

for fold_i, heldout_contig in enumerate(TRAIN_CONTIGS, 1):
    print()
    print("=" * 80)
    print(
        f"EVAL NATURAL FOLD {fold_i}/{len(TRAIN_CONTIGS)} "
        f"{heldout_contig}"
    )
    print("=" * 80)

    heldout_tiles = (
        train_tiles[
            train_tiles["contig"].astype(str) == heldout_contig
        ]
        .reset_index(drop=True)
    )

    head = cv_heads[heldout_contig]
    states = make_eval_state()

    started = time.perf_counter()
    positions_seen = 0

    with torch.no_grad():
        for row_i in range(len(heldout_tiles)):
            selected = heldout_tiles.iloc[row_i:row_i + 1]
            batch = train_generator.make_batch(selected)

            features, old_logit, _ = extract_backbone_and_old_outputs(
                frozen_model,
                batch,
            )

            plus_count = (
                batch.count[0, 0, :]
                .numpy().astype(np.float32, copy=False)
            )
            minus_count_reverse = (
                batch.count[0, 1, :]
                .flip(-1)
                .numpy().astype(np.float32, copy=False)
            )

            plus_mask = (
                batch.mask[0, 0, :]
                .numpy().astype(bool, copy=False)
            )
            minus_mask_reverse = (
                batch.mask[0, 1, :]
                .flip(-1)
                .numpy().astype(bool, copy=False)
            )

            for orient_i, (count, mask) in enumerate(
                (
                    (plus_count, plus_mask),
                    (minus_count_reverse, minus_mask_reverse),
                )
            ):
                valid_index = np.flatnonzero(mask)

                valid_feature = features[orient_i, valid_index]
                valid_old_logit = old_logit[orient_i, valid_index]
                valid_count = count[valid_index]

                positive = valid_count > 0
                correction = correction_raw(valid_feature, head)

                for alpha in ALPHAS:
                    final_logit = valid_old_logit + alpha * correction
                    probability = sigmoid_numpy(final_logit)
                    score_bin = probability_bin_5dp(probability)

                    state = states[alpha]

                    update_histogram(
                        state["total_hist"],
                        score_bin,
                    )

                    if positive.any():
                        update_histogram(
                            state["positive_hist"],
                            score_bin[positive],
                        )
                        state["positive_score"].append(
                            probability[positive].copy()
                        )
                        state["positive_count"].append(
                            valid_count[positive].copy()
                        )

                positions_seen += int(mask.sum())

            del (
                batch,
                selected,
                features,
                old_logit,
                plus_count,
                minus_count_reverse,
                plus_mask,
                minus_mask_reverse,
            )

            if (
                row_i == 0
                or (row_i + 1) % 250 == 0
                or row_i + 1 == len(heldout_tiles)
            ):
                elapsed = time.perf_counter() - started
                print(
                    f"tiles={row_i + 1:>5}/{len(heldout_tiles)} "
                    f"positions={positions_seen:>12,} "
                    f"positions/s={positions_seen / elapsed:,.0f}"
                )

            if (row_i + 1) % 50 == 0:
                gc.collect()

    # Diagnostic LOW/HIGH TP cohorts are defined from alpha=0 on heldout labels.
    baseline_positive_score = np.concatenate(
        states[0.0]["positive_score"]
    )

    low_cutoff_eval = float(
        np.quantile(
            baseline_positive_score,
            LOW_TP_QUANTILE,
        )
    )
    high_cutoff_eval = float(
        np.quantile(
            baseline_positive_score,
            0.75,
        )
    )

    for alpha in ALPHAS:
        state = states[alpha]

        positive_score = np.concatenate(
            state["positive_score"]
        )
        positive_count = np.concatenate(
            state["positive_count"]
        )

        baseline_low_mask = baseline_positive_score <= low_cutoff_eval
        baseline_high_mask = baseline_positive_score >= high_cutoff_eval

        cv_rows.append(
            {
                "heldout_contig": heldout_contig,
                "alpha": alpha,
                "average_precision_5dp": ap_from_histograms(
                    state["positive_hist"],
                    state["total_hist"],
                ),
                "positive_spearman_raw": dense_rank_pearson(
                    positive_score,
                    positive_count,
                ),
                "low_tp_mean_score": float(
                    positive_score[baseline_low_mask].mean()
                ),
                "high_tp_mean_score": float(
                    positive_score[baseline_high_mask].mean()
                ),
                "positive_n": int(len(positive_count)),
            }
        )

cv_results = pd.DataFrame(cv_rows)
cv_results.to_csv(CV_RESULTS_PATH, index=False)

display(cv_results)
print("Saved:", CV_RESULTS_PATH)



EVAL NATURAL FOLD 1/9 NC_064035.1
tiles=    1/18918 positions=         498 positions/s=12,218
tiles=  250/18918 positions=     374,670 positions/s=204,793
tiles=  500/18918 positions=     811,248 positions/s=218,961
tiles=  750/18918 positions=   1,209,988 positions/s=217,854
tiles= 1000/18918 positions=   1,606,742 positions/s=217,209
tiles= 1250/18918 positions=   2,025,652 positions/s=219,223
tiles= 1500/18918 positions=   2,447,804 positions/s=221,046
tiles= 1750/18918 positions=   2,875,410 positions/s=222,304
tiles= 2000/18918 positions=   3,276,578 positions/s=221,882
tiles= 2250/18918 positions=   3,727,040 positions/s=224,140
tiles= 2500/18918 positions=   4,184,270 positions/s=226,004
tiles= 2750/18918 positions=   4,626,284 positions/s=227,074
tiles= 3000/18918 positions=   5,062,232 positions/s=227,528
tiles= 3250/18918 positions=   5,524,502 positions/s=228,809
tiles= 3500/18918 positions=   5,960,950 positions/s=229,133
tiles= 3750/18918 positions=   6,421,794 positions/

,heldout_contig,alpha,average_precision_5dp,positive_spearman_raw,low_tp_mean_score,high_tp_mean_score,positive_n
0,NC_064035.1,0.00,0.040684,0.187394,0.335053,0.987871,30987
1,NC_064035.1,0.10,0.040763,0.187439,0.353474,0.990380,30987
2,NC_064035.1,0.25,0.040812,0.187404,0.381277,0.993195,30987
3,NC_064035.1,0.50,0.040780,0.187155,0.427197,0.996161,30987
4,NC_064035.1,1.00,0.040186,0.185595,0.512702,0.998752,30987
5,NC_064036.1,0.00,0.042699,0.202202,0.344143,0.989242,26323
6,NC_064036.1,0.10,0.042836,0.202286,0.362611,0.991454,26323
7,NC_064036.1,0.25,0.042963,0.202167,0.390393,0.993939,26323
8,NC_064036.1,0.50,0.042993,0.201872,0.436027,0.996564,26323
9,NC_064036.1,1.00,0.042421,0.199873,0.520199,0.998868,26323


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001\cross_contig_cv.csv


In [10]:
cv_summary = (
    cv_results
    .groupby("alpha", as_index=False)
    .agg(
        mean_AP=("average_precision_5dp", "mean"),
        min_AP=("average_precision_5dp", "min"),
        max_AP=("average_precision_5dp", "max"),
        mean_positive_spearman=("positive_spearman_raw", "mean"),
        mean_low_tp_score=("low_tp_mean_score", "mean"),
        mean_high_tp_score=("high_tp_mean_score", "mean"),
    )
    .sort_values(
        ["mean_AP", "min_AP"],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(cv_summary)

BEST_ALPHA = float(cv_summary.iloc[0]["alpha"])

CV_SUMMARY_PATH = RUN_DIR / "cross_contig_cv_summary.csv"
cv_summary.to_csv(CV_SUMMARY_PATH, index=False)

print("SELECTED ALPHA:", BEST_ALPHA)

if BEST_ALPHA == 0.0:
    print(
        "CV выбрал alpha=0: пока residual correction "
        "не улучшает natural-background ranking."
    )


,alpha,mean_AP,min_AP,max_AP,mean_positive_spearman,mean_low_tp_score,mean_high_tp_score
0,0.10,0.038910,0.031372,0.045825,0.189229,0.351196,0.991156
1,0.25,0.038907,0.031330,0.045781,0.189006,0.378473,0.993762
2,0.00,0.038850,0.031341,0.045719,0.189322,0.333111,0.988827
3,0.50,0.038762,0.031287,0.045589,0.188470,0.423476,0.996495
4,1.00,0.037875,0.030781,0.044210,0.186654,0.507169,0.998868


SELECTED ALPHA: 0.1


## 5. Final residual correction на всех 9 train-contigs


In [11]:
FINAL_HEAD = train_residual_correction(
    train_pos_X=pos_X,
    train_pos_score=pos_old_score,
    train_random_X=random_X,
    train_random_score=random_old_score,
    seed=999_038,
)

FINAL_HEAD_PATH = RUN_DIR / "final_lowtp_residual_head.npz"

np.savez(
    FINAL_HEAD_PATH,
    w_raw=FINAL_HEAD["w_raw"],
    b_raw=np.array([FINAL_HEAD["b_raw"]], dtype=np.float32),
    alpha=np.array([BEST_ALPHA], dtype=np.float32),
    low_cutoff=np.array([FINAL_HEAD["low_cutoff"]], dtype=np.float32),
)

print("Selected alpha:", BEST_ALPHA)
print("Final residual head:", FINAL_HEAD_PATH)
print(
    {
        "LOW_TP": FINAL_HEAD["n_low_tp"],
        "matched_TN": FINAL_HEAD["n_matched_tn"],
        "random_TN": FINAL_HEAD["n_random_tn"],
        "train_LOW_TP_cutoff": FINAL_HEAD["low_cutoff"],
    }
)


Selected alpha: 0.1
Final residual head: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001\final_lowtp_residual_head.npz
{'LOW_TP': 42611, 'matched_TN': 85222, 'random_TN': 170444, 'train_LOW_TP_cutoff': 0.6023049354553223}


## 6. Один полный validation

Оцениваем только после выбора α.

Сохраняем:

- overall AP baseline vs residual;
- AP по каждому validation contig;
- Spearman;
- LOW_TP recovery в top-1/5/10%;
- HIGH_TP preservation;
- new FP / recovered LOW_TP.


In [12]:
BUDGETS = (0.01, 0.05, 0.10)


def threshold_for_top_fraction(total_hist, fraction):
    total = int(total_hist.sum())
    target = int(math.ceil(fraction * total))

    desc = total_hist[::-1]
    cumulative = np.cumsum(desc, dtype=np.int64)

    idx = int(
        np.searchsorted(
            cumulative,
            target,
            side="left",
        )
    )

    bin_index = SUBMISSION_LEVELS - 1 - idx
    selected = int(cumulative[idx])

    return {
        "bin_index": int(bin_index),
        "threshold": float(bin_index / 100_000.0),
        "selected": selected,
        "selected_fraction": float(selected / total),
    }


def budget_stats(total_hist, positive_hist, fraction):
    threshold = threshold_for_top_fraction(
        total_hist,
        fraction,
    )

    b = threshold["bin_index"]

    selected = int(total_hist[b:].sum())
    tp = int(positive_hist[b:].sum())
    fp = selected - tp

    positives = int(positive_hist.sum())

    return {
        **threshold,
        "tp": tp,
        "fp": fp,
        "precision": float(tp / max(selected, 1)),
        "recall": float(tp / max(positives, 1)),
    }


def new_validation_state():
    return {
        "baseline_total_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "baseline_positive_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "residual_total_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "residual_positive_hist": np.zeros(
            SUBMISSION_LEVELS,
            dtype=np.int64,
        ),
        "positive_baseline_score": [],
        "positive_residual_score": [],
        "positive_intensity": [],
        "positive_count": [],
    }


In [13]:
segment_names = ["overall", *VALIDATION_CONTIGS]
validation_states = {
    segment: new_validation_state()
    for segment in segment_names
}

started = time.perf_counter()
positions_seen = 0

frozen_model.eval()

with torch.no_grad():
    for row_i in range(len(validation_tiles)):
        selected = validation_tiles.iloc[row_i:row_i + 1]
        contig = str(selected.iloc[0]["contig"])

        batch = validation_generator.make_batch(selected)

        features, old_logit, old_intensity = extract_backbone_and_old_outputs(
            frozen_model,
            batch,
        )

        plus_count = (
            batch.count[0, 0, :]
            .numpy().astype(np.float32, copy=False)
        )
        minus_count_reverse = (
            batch.count[0, 1, :]
            .flip(-1)
            .numpy().astype(np.float32, copy=False)
        )

        plus_mask = (
            batch.mask[0, 0, :]
            .numpy().astype(bool, copy=False)
        )
        minus_mask_reverse = (
            batch.mask[0, 1, :]
            .flip(-1)
            .numpy().astype(bool, copy=False)
        )

        for orient_i, (count, mask) in enumerate(
            (
                (plus_count, plus_mask),
                (minus_count_reverse, minus_mask_reverse),
            )
        ):
            valid_index = np.flatnonzero(mask)

            valid_feature = features[orient_i, valid_index]
            valid_old_logit = old_logit[orient_i, valid_index]
            valid_intensity = old_intensity[orient_i, valid_index]
            valid_count = count[valid_index]

            positive = valid_count > 0

            correction = correction_raw(
                valid_feature,
                FINAL_HEAD,
            )

            baseline_probability = sigmoid_numpy(valid_old_logit)

            residual_probability = sigmoid_numpy(
                valid_old_logit
                + BEST_ALPHA * correction
            )

            baseline_bin = probability_bin_5dp(baseline_probability)
            residual_bin = probability_bin_5dp(residual_probability)

            for segment in ("overall", contig):
                state = validation_states[segment]

                update_histogram(
                    state["baseline_total_hist"],
                    baseline_bin,
                )
                update_histogram(
                    state["residual_total_hist"],
                    residual_bin,
                )

                if positive.any():
                    update_histogram(
                        state["baseline_positive_hist"],
                        baseline_bin[positive],
                    )
                    update_histogram(
                        state["residual_positive_hist"],
                        residual_bin[positive],
                    )

                    state["positive_baseline_score"].append(
                        baseline_probability[positive].copy()
                    )
                    state["positive_residual_score"].append(
                        residual_probability[positive].copy()
                    )
                    state["positive_intensity"].append(
                        valid_intensity[positive].copy()
                    )
                    state["positive_count"].append(
                        valid_count[positive].copy()
                    )

            positions_seen += int(mask.sum())

        del (
            batch,
            selected,
            features,
            old_logit,
            old_intensity,
            plus_count,
            minus_count_reverse,
            plus_mask,
            minus_mask_reverse,
        )

        if (
            row_i == 0
            or (row_i + 1) % 100 == 0
            or row_i + 1 == len(validation_tiles)
        ):
            elapsed = time.perf_counter() - started
            print(
                f"tiles={row_i + 1:>6}/{len(validation_tiles)} "
                f"positions={positions_seen:>13,} "
                f"positions/s={positions_seen / elapsed:,.0f}"
            )

        if (row_i + 1) % 25 == 0:
            gc.collect()

print("FULL VALIDATION FINISHED")


tiles=     1/47743 positions=          210 positions/s=5,038
tiles=   100/47743 positions=      125,828 positions/s=153,295
tiles=   200/47743 positions=      269,398 positions/s=160,859
tiles=   300/47743 positions=      403,094 positions/s=158,731
tiles=   400/47743 positions=      552,594 positions/s=162,339
tiles=   500/47743 positions=      708,204 positions/s=165,241
tiles=   600/47743 positions=      871,458 positions/s=168,810
tiles=   700/47743 positions=    1,020,090 positions/s=169,258
tiles=   800/47743 positions=    1,185,336 positions/s=171,862
tiles=   900/47743 positions=    1,346,300 positions/s=173,268
tiles=  1000/47743 positions=    1,519,462 positions/s=175,858
tiles=  1100/47743 positions=    1,700,978 positions/s=178,675
tiles=  1200/47743 positions=    1,853,300 positions/s=178,587
tiles=  1300/47743 positions=    2,017,896 positions/s=179,457
tiles=  1400/47743 positions=    2,193,246 positions/s=180,908
tiles=  1500/47743 positions=    2,381,142 positions/s=18

In [14]:
metric_rows = []
positive_cache = {}

for segment, state in validation_states.items():
    baseline_positive = np.concatenate(
        state["positive_baseline_score"]
    )
    residual_positive = np.concatenate(
        state["positive_residual_score"]
    )
    intensity_positive = np.concatenate(
        state["positive_intensity"]
    )
    positive_count = np.concatenate(
        state["positive_count"]
    )

    baseline_ap = ap_from_histograms(
        state["baseline_positive_hist"],
        state["baseline_total_hist"],
    )
    residual_ap = ap_from_histograms(
        state["residual_positive_hist"],
        state["residual_total_hist"],
    )

    metric_rows.append(
        {
            "segment": segment,
            "positive_n": int(len(positive_count)),
            "baseline_AP_5dp": baseline_ap,
            "residual_AP_5dp": residual_ap,
            "delta_AP": residual_ap - baseline_ap,
            "baseline_presence_spearman_raw": dense_rank_pearson(
                baseline_positive,
                positive_count,
            ),
            "residual_presence_spearman_raw": dense_rank_pearson(
                residual_positive,
                positive_count,
            ),
            "frozen_intensity_spearman_raw": dense_rank_pearson(
                intensity_positive,
                positive_count,
            ),
        }
    )

    positive_cache[segment] = {
        "baseline": baseline_positive,
        "residual": residual_positive,
        "count": positive_count,
    }

final_metrics = (
    pd.DataFrame(metric_rows)
    .sort_values("segment")
    .reset_index(drop=True)
)

FINAL_METRICS_PATH = RUN_DIR / "final_validation_metrics.csv"
final_metrics.to_csv(FINAL_METRICS_PATH, index=False)

display(final_metrics)
print("Saved:", FINAL_METRICS_PATH)


,segment,positive_n,baseline_AP_5dp,residual_AP_5dp,delta_AP,baseline_presence_spearman_raw,residual_presence_spearman_raw,frozen_intensity_spearman_raw
0,NC_064034.1,23529,0.033783,0.033804,0.000022,0.191207,0.191239,0.203072
1,NC_064041.1,13984,0.028973,0.029041,0.000068,0.221014,0.221324,0.234122
2,NC_064042.1,18692,0.038690,0.038827,0.000136,0.206500,0.206593,0.220363
3,overall,56205,0.033997,0.034073,0.000076,0.177041,0.177058,0.188709


Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001\final_validation_metrics.csv


In [15]:
overall_state = validation_states["overall"]
overall_positive = positive_cache["overall"]

baseline_positive = overall_positive["baseline"]
residual_positive = overall_positive["residual"]

low_cutoff = float(
    np.quantile(
        baseline_positive,
        LOW_TP_QUANTILE,
    )
)
high_cutoff = float(
    np.quantile(
        baseline_positive,
        0.75,
    )
)

low_mask = baseline_positive <= low_cutoff
high_mask = baseline_positive >= high_cutoff

budget_rows = []

for fraction in BUDGETS:
    baseline_stats = budget_stats(
        overall_state["baseline_total_hist"],
        overall_state["baseline_positive_hist"],
        fraction,
    )
    residual_stats = budget_stats(
        overall_state["residual_total_hist"],
        overall_state["residual_positive_hist"],
        fraction,
    )

    baseline_threshold = baseline_stats["threshold"]
    residual_threshold = residual_stats["threshold"]

    baseline_low_selected = int(
        np.sum(
            baseline_positive[low_mask]
            >= baseline_threshold
        )
    )
    residual_low_selected = int(
        np.sum(
            residual_positive[low_mask]
            >= residual_threshold
        )
    )

    baseline_high_selected = int(
        np.sum(
            baseline_positive[high_mask]
            >= baseline_threshold
        )
    )
    residual_high_selected = int(
        np.sum(
            residual_positive[high_mask]
            >= residual_threshold
        )
    )

    recovered_low_tp = (
        residual_low_selected
        - baseline_low_selected
    )

    new_fp = (
        residual_stats["fp"]
        - baseline_stats["fp"]
    )

    if recovered_low_tp > 0:
        fp_per_recovered = float(
            new_fp / recovered_low_tp
        )
    else:
        fp_per_recovered = float("nan")

    budget_rows.append(
        {
            "top_fraction": fraction,

            "baseline_precision": baseline_stats["precision"],
            "baseline_recall": baseline_stats["recall"],
            "baseline_FP": baseline_stats["fp"],

            "residual_precision": residual_stats["precision"],
            "residual_recall": residual_stats["recall"],
            "residual_FP": residual_stats["fp"],

            "baseline_LOW_TP_selected": baseline_low_selected,
            "residual_LOW_TP_selected": residual_low_selected,
            "recovered_LOW_TP": recovered_low_tp,

            "baseline_LOW_TP_recall": float(
                baseline_low_selected / max(int(low_mask.sum()), 1)
            ),
            "residual_LOW_TP_recall": float(
                residual_low_selected / max(int(low_mask.sum()), 1)
            ),

            "baseline_HIGH_TP_selected": baseline_high_selected,
            "residual_HIGH_TP_selected": residual_high_selected,
            "baseline_HIGH_TP_recall": float(
                baseline_high_selected / max(int(high_mask.sum()), 1)
            ),
            "residual_HIGH_TP_recall": float(
                residual_high_selected / max(int(high_mask.sum()), 1)
            ),

            "delta_FP": new_fp,
            "new_FP_per_recovered_LOW_TP": fp_per_recovered,
        }
    )

budget_diagnostics = pd.DataFrame(budget_rows)

BUDGET_PATH = RUN_DIR / "lowtp_budget_diagnostics.csv"
budget_diagnostics.to_csv(BUDGET_PATH, index=False)

display(budget_diagnostics)

print("Validation LOW_TP cutoff:", low_cutoff)
print("Validation LOW_TP n:", int(low_mask.sum()))
print("Validation HIGH_TP cutoff:", high_cutoff)
print("Validation HIGH_TP n:", int(high_mask.sum()))
print("Saved:", BUDGET_PATH)


,top_fraction,baseline_precision,baseline_recall,baseline_FP,residual_precision,residual_recall,residual_FP,baseline_LOW_TP_selected,residual_LOW_TP_selected,recovered_LOW_TP,baseline_LOW_TP_recall,residual_LOW_TP_recall,baseline_HIGH_TP_selected,residual_HIGH_TP_selected,baseline_HIGH_TP_recall,residual_HIGH_TP_recall,delta_FP,new_FP_per_recovered_LOW_TP
0,0.01,0.022838,0.340539,818936,0.022877,0.341126,818902,0,0,0,0.0,0.000000,14052,14052,1.0,1.0,-34,NaN
1,0.05,0.008167,0.608914,4156320,0.008191,0.610693,4156175,0,0,0,0.0,0.000000,14052,14052,1.0,1.0,-145,NaN
2,0.10,0.004963,0.740041,8339283,0.004972,0.741358,8339220,0,4,4,0.0,0.000285,14052,14052,1.0,1.0,-63,-15.75


Validation LOW_TP cutoff: 0.5802421569824219
Validation LOW_TP n: 14052
Validation HIGH_TP cutoff: 0.9693856835365295
Validation HIGH_TP n: 14052
Saved: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001\lowtp_budget_diagnostics.csv


## 7. Итог

Интерпретация:

```text
если BEST_ALPHA = 0
→ 64-D representation не даёт полезной residual correction
  для LOW_TP на natural background.

если alpha > 0 и AP растёт
→ в 64-D features есть дополнительный LOW_TP signal,
  который старая head не использует.

если LOW_TP растёт, но FP/LOW_TP огромен
→ локальный signal есть, но global ranking экономически плох.

если AP растёт без потери HIGH_TP
→ residual idea стоит переносить в end-to-end objective.
```


In [16]:
WIDTH64_REFERENCE_AP = 0.03399689188641901

overall_row = (
    final_metrics[
        final_metrics["segment"] == "overall"
    ]
    .iloc[0]
)

contig_rows = final_metrics[
    final_metrics["segment"] != "overall"
]

improved_contigs = int(
    np.sum(contig_rows["delta_AP"] > 0)
)

summary = {
    "experiment": {
        "id": EXPERIMENT_ID,
        "hypothesis": (
            "LOW_TP can receive a useful score-matched residual correction "
            "from frozen WIDTH64 features without replacing EXP036 ranking."
        ),
        "backbone": "frozen WIDTH64 RF1029",
        "residual_head": "linear 64->1",
        "selected_alpha": BEST_ALPHA,
        "alpha_grid": list(ALPHAS),
        "low_tp_quantile": LOW_TP_QUANTILE,
        "matched_tn_per_lowtp": MATCHED_TN_PER_LOWTP,
        "random_tn_per_lowtp": RANDOM_TN_PER_LOWTP,
        "cv_selection_metric": (
            "mean full natural-background AP on held-out train contigs"
        ),
    },
    "validation": {
        "reference_width64_AP": WIDTH64_REFERENCE_AP,
        "baseline_recomputed_AP": float(
            overall_row["baseline_AP_5dp"]
        ),
        "residual_AP": float(
            overall_row["residual_AP_5dp"]
        ),
        "delta_AP": float(
            overall_row["delta_AP"]
        ),
        "improved_validation_contigs": improved_contigs,
        "validation_contigs_total": int(len(contig_rows)),
        "baseline_presence_spearman_raw": float(
            overall_row["baseline_presence_spearman_raw"]
        ),
        "residual_presence_spearman_raw": float(
            overall_row["residual_presence_spearman_raw"]
        ),
        "frozen_intensity_spearman_raw": float(
            overall_row["frozen_intensity_spearman_raw"]
        ),
    },
    "files": {
        "cross_contig_cv": str(CV_RESULTS_PATH),
        "cross_contig_cv_summary": str(CV_SUMMARY_PATH),
        "final_residual_head": str(FINAL_HEAD_PATH),
        "final_validation_metrics": str(FINAL_METRICS_PATH),
        "lowtp_budget_diagnostics": str(BUDGET_PATH),
    },
}

RESULT_JSON = RUN_DIR / "EXP038_RESULT.json"

RESULT_JSON.write_text(
    json.dumps(
        summary,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("=" * 80)
print("EXP038 COMPLETE")
print("=" * 80)
print()
print(
    json.dumps(
        summary["validation"],
        ensure_ascii=False,
        indent=2,
    )
)
print()
print("Send me:")
print(RESULT_JSON)
print(CV_SUMMARY_PATH)
print(BUDGET_PATH)


EXP038 COMPLETE

{
  "reference_width64_AP": 0.03399689188641901,
  "baseline_recomputed_AP": 0.03399689188641901,
  "residual_AP": 0.03407284928345632,
  "delta_AP": 7.595739703731424e-05,
  "improved_validation_contigs": 3,
  "validation_contigs_total": 3,
  "baseline_presence_spearman_raw": 0.17704136669635773,
  "residual_presence_spearman_raw": 0.17705833911895752,
  "frozen_intensity_spearman_raw": 0.18870866298675537
}

Send me:
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001\EXP038_RESULT.json
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001\cross_contig_cv_summary.csv
D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001\lowtp_budget_diagnostics.csv


## 8. Optional registration

Старый reporter ожидает `average_precision` / `epic_spearman`
и заранее существующую markdown card.

Compatibility-layer включён.


In [17]:
SAVE_EXPERIMENT = False

REPORT_METRICS_PATH = RUN_DIR / "validation_metrics_report.csv"

report_metrics = (
    final_metrics[
        [
            "segment",
            "residual_AP_5dp",
            "residual_presence_spearman_raw",
        ]
    ]
    .rename(
        columns={
            "residual_AP_5dp": "average_precision",
            "residual_presence_spearman_raw": "epic_spearman",
        }
    )
)

report_metrics.to_csv(
    REPORT_METRICS_PATH,
    index=False,
)

CARD_PATH = (
    PROJECT_ROOT
    / "cnn_exp"
    / f"{EXPERIMENT_ID}.md"
)

CARD_PATH.parent.mkdir(parents=True, exist_ok=True)

if not CARD_PATH.exists():
    CARD_PATH.write_text(
        f'''---
id: {EXPERIMENT_ID}
type: experiment
experiment_type: cnn-candidate
status: completed
decision: review
date: 2026-09-24
seed: 42
split_version: contig_holdout_v1
primary_metric: average_precision
secondary_metric: epic_spearman
reference_experiment: CNN-EXP-036
tags:
  - ml/experiment
  - ml/cnn
  - ml/low-tp
  - ml/residual
---

# {EXPERIMENT_ID} — LOW_TP score-matched residual test

← [[cnn_exp/_index.md|Реестр CNN]]

## Pre-registration

- **Гипотеза:** frozen WIDTH64 features содержат residual LOW_TP signal,
  который можно добавить к старому EXP036 logit, не разрушая baseline ranking.
- **Единственное изменение:** linear residual correction к old presence logit.
- **Что заморожено:** WIDTH64 backbone, RF1029, old presence-head,
  old intensity-head, split.
- **Reference:** [[cnn_exp/CNN-EXP-036.md|CNN-EXP-036]].

## Автоматический отчёт

<!-- auto:cnn-experiment-report:start -->

<!-- auto:cnn-experiment-report:end -->

## Ручные заметки

-
''',
        encoding="utf-8",
    )

print("Reporter metrics:", REPORT_METRICS_PATH)
print("Card:", CARD_PATH)


Reporter metrics: D:\Projects\EPIC\EPIC\artifacts\experiments\CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL\run_001\validation_metrics_report.csv
Card: D:\Projects\EPIC\EPIC\cnn_exp\CNN-EXP-038.md


In [18]:
if SAVE_EXPERIMENT:
    from ml_project.cnn_experiment_report import (
        finalize_cnn_experiment,
    )

    reference_report_metrics = (
        EXP036_RUN
        / "validation_metrics_report.csv"
    )

    assert reference_report_metrics.is_file(), (
        "Нет EXP036 validation_metrics_report.csv. "
        "Сначала выполни compatibility adapter в EXP036."
    )

    finalization = finalize_cnn_experiment(
        project_root=PROJECT_ROOT,
        experiment_id=EXPERIMENT_ID,
        title="LOW_TP score-matched residual test",
        hypothesis=(
            "A small linear residual correction from frozen WIDTH64 features "
            "can recover LOW_TP against score-matched natural TN without "
            "replacing the EXP036 presence ranking."
        ),
        notebook_path=(
            "notebooks/experiments/CNN_experiments/"
            "CNN_EXP038_LOWTP_SCOREMATCHED_RESIDUAL.ipynb"
        ),
        artifact_dir=RUN_DIR,
        training_history=EXP036_RUN / "training_history.csv",
        candidate_metrics=REPORT_METRICS_PATH,
        reference_experiment="CNN-EXP-036",
        reference_metrics=reference_report_metrics,
        decision="review",
        notes=(
            "Frozen WIDTH64. Residual linear correction. "
            "alpha selected by train-only cross-contig natural-background AP."
        ),
    )

    print("Obsidian:", finalization.card_path)
    print("Summary:", finalization.summary_path)
    print("Plots:", finalization.plot_dir)

else:
    print(
        "SAVE_EXPERIMENT=False. "
        "Сначала пришли EXP038_RESULT.json, "
        "cross_contig_cv_summary.csv и "
        "lowtp_budget_diagnostics.csv."
    )


SAVE_EXPERIMENT=False. Сначала пришли EXP038_RESULT.json, cross_contig_cv_summary.csv и lowtp_budget_diagnostics.csv.
